# Music Genre Classification with Google LEAF
Train a CNN classifier using PyTorch and Google's Learnable Audio Frontend (LEAF) on the GTZAN dataset.
Make sure your runtime is set to **T4 GPU**.

In [ ]:
!pip install torch torchaudio soundfile scikit-learn
!git clone https://github.com/SarthakYadav/leaf-pytorch.git

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import soundfile as sf
import os
import sys
import torchaudio
import urllib.request
import tarfile

sys.path.append(os.path.join(os.getcwd(), 'leaf-pytorch'))
from leaf_pytorch.frontend import Leaf

In [ ]:
def ensure_gtzan_downloaded(root="data"):
    gtzan_dir = os.path.join(root, "gtzan")
    os.makedirs(gtzan_dir, exist_ok=True)
    tar_path = os.path.join(gtzan_dir, "genres.tar.gz")
    if not os.path.exists(tar_path):
        print("Downloading GTZAN from HuggingFace mirror (1.2GB)... This might take a few minutes.")
        url = "https://huggingface.co/datasets/marsyas/gtzan/resolve/main/data/genres.tar.gz"
        urllib.request.urlretrieve(url, tar_path)
        print("Download complete. Extracting...")
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(path=gtzan_dir)
        print("Extraction complete.")
    return root

class GTZANDataset(Dataset):
    def __init__(self, root="data/gtzan/genres", sample_rate=16000, duration=5.0):
        self.root = root
        self.target_sample_rate = sample_rate
        self.max_length = int(sample_rate * duration)
        
        self.genres = [
            "blues", "classical", "country", "disco", "hiphop",
            "jazz", "metal", "pop", "reggae", "rock"
        ]
        self.genre_to_idx = {g: i for i, g in enumerate(self.genres)}
        self.resamplers = {}
        
        self.files = []
        for genre in self.genres:
            genre_dir = os.path.join(root, genre)
            if not os.path.exists(genre_dir): continue
            for f in os.listdir(genre_dir):
                if f.endswith('.wav') or f.endswith('.au'):
                    self.files.append((os.path.join(genre_dir, f), genre))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path, label = self.files[idx]
        try:
            waveform, sr = torchaudio.load(file_path)
        except Exception:
            waveform = torch.zeros(1, self.max_length)
            sr = self.target_sample_rate
        
        # Optimized resampling logic
        if sr != self.target_sample_rate:
            if sr not in self.resamplers:
                # Cache the resampler for this sample rate
                self.resamplers[sr] = torchaudio.transforms.Resample(
                    orig_freq=sr, 
                    new_freq=self.target_sample_rate
                )
            # Use the cached resampler
            waveform = self.resamplers[sr](waveform)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        if waveform.shape[1] > self.max_length:
            waveform = waveform[:, :self.max_length]
        elif waveform.shape[1] < self.max_length:
            pad_amount = self.max_length - waveform.shape[1]
            waveform = F.pad(waveform, (0, pad_amount))
            
        label_idx = self.genre_to_idx[label]
        return waveform, label_idx

In [ ]:
class AudioClassifier(nn.Module):
    def __init__(self, num_classes=10, sample_rate=16000, n_filters=40):
        super(ImprovedAudioClassifier, self).__init__()
        
        # 1. Learnable Audio Frontend
        self.leaf = Leaf(
            sample_rate=sample_rate, 
            n_filters=n_filters,
            init_min_freq=60.0,
            init_max_freq=7800.0
        )
        
        # 2. 2D CNN Backbone
        # Input shape from LEAF: (Batch, n_filters, Time)
        # We will unsqueeze it to: (Batch, 1, n_filters, Time) to treat it as a 2D image
        
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout2d(0.2)
        )
        
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout2d(0.3)
        )
        
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Dropout2d(0.3)
        )
        
        self.conv_block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)), # Pools (Freq, Time) down to 1x1
            nn.Dropout(0.4)
        )
        
        # 3. Classifier Head
        self.fc1 = nn.Linear(256, 128)
        self.dropout_fc = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # 1. LEAF
        x = self.leaf(x)  # Shape: (B, 40, Time)
        
        # 2. Add channel dimension for 2D Convolutions
        x = x.unsqueeze(1) # Shape: (B, 1, 40, Time)
        
        # 3. 2D Convolutions
        x = self.conv_block1(x)
        x = self.conv_block2(x)
        x = self.conv_block3(x)
        x = self.conv_block4(x)
        
        # 4. Flatten
        x = x.view(x.size(0), -1) # Shape: (B, 256)
        
        # 5. Fully Connected
        x = F.relu(self.fc1(x))
        x = self.dropout_fc(x)
        x = self.fc2(x)
        
        return x

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

root_dir = ensure_gtzan_downloaded()
full_dataset = GTZANDataset(root=os.path.join(root_dir, "gtzan/genres"))
print(f"Total dataset size: {len(full_dataset)}")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

model = AudioClassifier(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

scaler = torch.amp.GradScaler()

epochs = 30
print(f"Starting training for {epochs} epochs...")

for epoch in range(epochs):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        
        # ---> Use Mixed Precision (AMP) for Forward and Loss <---
        with torch.amp.autocast("cuda"):
            outputs = model(inputs)
            loss = criterion(outputs, targets)
        
        # ---> Scaled Backward Pass <---
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
    train_loss = total_loss / len(train_loader)
    train_acc = 100. * correct / total
    
    model.eval()
    val_loss = 0
    all_targets, all_preds = [], []
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            all_targets.extend(targets.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            
    val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(all_targets, all_preds) * 100.
    val_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%, F1: {val_f1:.4f}")



In [ ]:
# Save the trained model
torch.save(model.state_dict(), "music_genre_classifier.pth")
print("Model saved to music_genre_classifier.pth")